In [1]:
import pandas as pd
from enumerate import *

In [2]:
df = pd.read_csv("data/schemes.csv")
len(df)

10666

In [3]:
df.head()

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth
0,Scheme1,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,A,A,B,B,A,A
1,Scheme2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,A,A,B,B,A,B
2,Scheme3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,A,A,B,B,B,A
3,Scheme4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,A,A,B,B,B,B
4,Scheme5,1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5,A,A,B,A,A,A


In [4]:
score_columns: list[str] = list(Scores._fields)
df["score"] = df.apply(lambda row: row[score_columns].sum(), axis = 1)
df.head()

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
0,Scheme1,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,A,A,B,B,A,A,5.0
1,Scheme2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,A,A,B,B,A,B,4.0
2,Scheme3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,A,A,B,B,B,A,4.0
3,Scheme4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,A,A,B,B,B,B,5.0
4,Scheme5,1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5,A,A,B,A,A,A,6.0


In [5]:
df.score.max()

np.float64(6.25)

In [6]:
df.groupby("score")[["scheme"]].count().rename(columns = dict(scheme = "num_schemes"))

,num_schemes
score,
1.50,8
1.75,44
2.00,342
2.25,544
2.50,354
2.75,432
3.00,998
3.25,1560
3.50,1394


In [7]:
len(df[score_columns].drop_duplicates())

808

In [8]:
df[df.score == 6.25]

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25
7122,Scheme7123,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,B-D,C-D,B-D,B-D,B-D,6.25
7198,Scheme7199,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,C-A,C-D,C-A,C-A,C-A,6.25
9786,Scheme9787,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,D-B,C-D,D-B,D-B,D-B,6.25


In [9]:
distinct_scores: pd.DataFrame = (
    df.groupby(score_columns, as_index = False)
    .scheme.count()
    .sort_values("scheme", ascending = False) # type: ignore
)

distinct_scores.head()

,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,scheme
70,0.0,0.25,0.50,0.50,0.25,0.25,0.25,0.25,264
122,0.0,0.25,0.75,0.75,0.50,0.25,0.25,0.50,160
117,0.0,0.25,0.75,0.75,0.25,0.50,0.50,0.25,160
269,0.0,0.50,1.00,1.00,0.50,0.50,0.50,0.50,144
212,0.0,0.50,0.75,0.75,0.50,0.50,0.25,0.25,96


In [10]:
distinct_scores.scheme.describe()

count    808.000000
mean      13.200495
std       19.352451
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      264.000000
Name: scheme, dtype: float64

In [11]:
from collections import Counter
from functools import reduce

counts: dict[int, int] = Counter(distinct_scores.scheme)
print(counts)

reduce(lambda sum, curr: sum + curr[0] * curr[1], counts.items(), 0)

Counter({4: 159, 8: 152, 2: 151, 24: 48, 16: 45, 32: 32, 12: 31, 6: 30, 20: 29, 1: 26, 64: 16, 40: 12, 10: 10, 3: 10, 48: 8, 56: 6, 14: 6, 5: 5, 76: 4, 44: 4, 28: 4, 9: 3, 160: 2, 96: 2, 60: 2, 52: 2, 36: 2, 26: 2, 18: 2, 264: 1, 144: 1, 72: 1})


10666

In [12]:
{k: v for k, v in counts.items() if k % 2 == 1}

{9: 3, 5: 5, 3: 10, 1: 26}

In [13]:
len(counts)

32

In [14]:
print({2 ** i: sum(v for k, v in counts.items() if k % (2 ** i) == 0) for i in range(10)})

{1: 808, 2: 764, 4: 563, 8: 326, 16: 106, 32: 52, 64: 16, 128: 0, 256: 0, 512: 0}


In [15]:
odds: pd.DataFrame = distinct_scores[distinct_scores.scheme % 2 == 1]
odd_scores: list[Iterable[float]] = list(odds[score_columns].to_numpy())
score_to_multiplicity: dict[tuple[float, ...], int] = \
    {tuple(odd_scores[i]): odds.scheme.iloc[i] for i in range(len(odds))}

odd_df: pd.DataFrame = df[df.set_index(score_columns).index.isin(odd_scores)]

In [16]:
odd_df["multiplicity"] = 0
for i in odd_df.index:
    odd_df.loc[i, "multiplicity"] = \
        score_to_multiplicity[tuple(odd_df.loc[i, score_columns])] # type: ignore

pd.set_option("display.max_rows", None)
odd_df.sort_values(["multiplicity"] + score_columns, inplace = True)

In [17]:
odd_df = odd_df.reset_index(drop = True)
odd_df["group_id"] = odd_df.groupby(score_columns, sort = False).ngroup()
odd_df

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score,multiplicity,group_id
0,Scheme3794,0.0,0.00,1.0,1.0,0.00,0.50,0.50,0.00,A-B,A-B,C-D,C-D,D-C,B-A,3.00,1,0
1,Scheme3758,0.0,0.00,1.0,1.0,0.00,0.50,1.00,0.00,A-B,A-B,C-D,C-D,C-D,B-A,3.50,1,1
2,Scheme3791,0.0,0.00,1.0,1.0,0.00,1.00,0.50,0.00,A-B,A-B,C-D,C-D,D-C,A-B,3.50,1,2
3,Scheme3706,0.0,0.00,1.0,1.0,0.50,0.00,0.00,0.50,A-B,A-B,C-D,C-D,B-A,D-C,3.00,1,3
4,Scheme3766,0.0,0.50,1.0,1.0,0.00,0.00,1.00,0.50,A-B,A-B,C-D,C-D,C-D,D-C,4.00,1,4
5,Scheme3695,0.0,0.50,1.0,1.0,0.50,1.00,0.00,0.00,A-B,A-B,C-D,C-D,B-A,A-B,4.00,1,5
6,Scheme3802,0.0,1.00,1.0,1.0,0.00,0.00,0.50,0.50,A-B,A-B,C-D,C-D,D-C,D-C,4.00,1,6
7,Scheme3698,0.0,1.00,1.0,1.0,0.50,0.50,0.00,0.00,A-B,A-B,C-D,C-D,B-A,B-A,4.00,1,7
8,Scheme10085,0.5,0.25,0.5,1.0,0.25,1.00,0.50,0.50,A-B,D-C,C-D,C-D,C-D,D-C,4.50,1,8
9,Scheme8408,0.5,0.25,0.5,1.0,0.50,0.75,0.25,0.75,A-B,C-D,C-D,D-C,C-D,D-C,4.50,1,9


In [18]:
groups_df = odd_df.drop_duplicates(score_columns).reset_index()[score_columns + ["score", "multiplicity"]]
groups_df

,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,score,multiplicity
0,0.0,0.00,1.0,1.0,0.00,0.50,0.50,0.00,3.00,1
1,0.0,0.00,1.0,1.0,0.00,0.50,1.00,0.00,3.50,1
2,0.0,0.00,1.0,1.0,0.00,1.00,0.50,0.00,3.50,1
3,0.0,0.00,1.0,1.0,0.50,0.00,0.00,0.50,3.00,1
4,0.0,0.50,1.0,1.0,0.00,0.00,1.00,0.50,4.00,1
5,0.0,0.50,1.0,1.0,0.50,1.00,0.00,0.00,4.00,1
6,0.0,1.00,1.0,1.0,0.00,0.00,0.50,0.50,4.00,1
7,0.0,1.00,1.0,1.0,0.50,0.50,0.00,0.00,4.00,1
8,0.5,0.25,0.5,1.0,0.25,1.00,0.50,0.50,4.50,1
9,0.5,0.25,0.5,1.0,0.50,0.75,0.25,0.75,4.50,1


In [19]:
from helpers.pandas_util import *

class GenderSwapRow(NamedTuple):
    swapped_family_id: int
    original_group_id: int
    swapped_group_id: int
    multiplicity: int
    score: float

families: list[Family] = import_families_from_dataframe(odd_df)
gender_swap_data: list[GenderSwapRow] = []

for i, family in enumerate(families):
    swapped_family_id: int = families.index(family.gender_swap())
    original_group_id: int = odd_df.loc[i, "group_id"].item() # type: ignore
    swapped_group_id: int = odd_df.loc[swapped_family_id, "group_id"].item() # type: ignore
    multiplicity: int = groups_df.loc[original_group_id, "multiplicity"].item() # type: ignore
    assert multiplicity == groups_df.loc[swapped_group_id, "multiplicity"].item() # type: ignore
    score: float = groups_df.loc[original_group_id, "score"].item() # type: ignore
    assert score == groups_df.loc[swapped_group_id, "score"].item() # type: ignore
    gender_swap_data.append(GenderSwapRow(swapped_family_id, original_group_id, swapped_group_id, multiplicity, score))

gender_swap_df = build_df_from_named_tuples(gender_swap_data)
gender_swap_df.index.rename("original_family_id", inplace = True)

gender_swap_df

,swapped_family_id,original_group_id,swapped_group_id,multiplicity,score
original_family_id,,,,,
0,0,0,0,1,3.00
1,2,1,2,1,3.50
2,1,2,1,1,3.50
3,3,3,3,1,3.00
4,5,4,5,1,4.00
5,4,5,4,1,4.00
6,7,6,7,1,4.00
7,6,7,6,1,4.00
8,10,8,10,1,4.50


In [20]:
gender_swap_df[gender_swap_df.original_group_id == gender_swap_df.swapped_group_id]

,swapped_family_id,original_group_id,swapped_group_id,multiplicity,score
original_family_id,,,,,
0,0,0,0,1,3.0
3,3,3,3,1,3.0
56,56,36,36,5,2.0
57,57,36,36,5,2.0
58,59,36,36,5,2.0
59,58,36,36,5,2.0
60,60,36,36,5,2.0
81,81,41,41,9,2.0
82,84,41,41,9,2.0


In [21]:
gender_swap_df[gender_swap_df.index == gender_swap_df.swapped_family_id]

,swapped_family_id,original_group_id,swapped_group_id,multiplicity,score
original_family_id,,,,,
0,0,0,0,1,3.0
3,3,3,3,1,3.0
56,56,36,36,5,2.0
57,57,36,36,5,2.0
60,60,36,36,5,2.0
81,81,41,41,9,2.0
86,86,41,41,9,2.0
89,89,41,41,9,2.0


In [22]:
print(list(enumerate(distinct_scores.scheme.head(30))))

[(0, 264), (1, 160), (2, 160), (3, 144), (4, 96), (5, 96), (6, 76), (7, 76), (8, 76), (9, 76), (10, 72), (11, 64), (12, 64), (13, 64), (14, 64), (15, 64), (16, 64), (17, 64), (18, 64), (19, 64), (20, 64), (21, 64), (22, 64), (23, 64), (24, 64), (25, 64), (26, 64), (27, 60), (28, 60), (29, 56)]


In [23]:
schemes: list[type[Scheme]] = \
    import_schemes_from_dataframe(df.loc[:, Family._fields], df.scheme) # type: ignore

[scheme.__name__ for scheme in schemes[:5]]

['Scheme1', 'Scheme2', 'Scheme3', 'Scheme4', 'Scheme5']

In [24]:
target_family = Family(
    father_birth = "A-B",
    father_married = "A-D",
    mother_birth = "C-D",
    mother_married = "A-D",
    son_birth = "A-D",
    daughter_birth = "A-D"
)

if any(
    (target_index := i) == i and export_scheme_to_family(scheme) == target_family
    for i, scheme in enumerate(schemes)
):
    print(target_index)

5154


In [25]:
df.iloc[target_index]

scheme             Scheme5155
husband_wife              1.0
brother_sister            0.5
brother_brother          0.75
sister_sister            0.75
son_father               0.75
daughter_father          0.75
son_mother               0.75
daughter_mother          0.75
father_birth              A-B
father_married            A-D
mother_birth              C-D
mother_married            A-D
son_birth                 A-D
daughter_birth            A-D
score                     6.0
Name: 5154, dtype: object

In [26]:
df[df.score == 6]

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
4,Scheme5,1.0,0.5,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.0
5,Scheme6,1.0,0.5,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.0
4162,Scheme4163,1.0,0.5,1.00,0.50,1.00,0.50,1.00,0.50,A-B,A-B,C-D,A-B,A-B,A-B,6.0
5154,Scheme5155,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-D,C-D,A-D,A-D,A-D,6.0
6502,Scheme6503,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,B-C,C-D,B-C,B-C,B-C,6.0
7818,Scheme7819,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,C-B,C-D,C-B,C-B,C-B,6.0
8330,Scheme8331,1.0,0.5,0.50,1.00,0.50,1.00,0.50,1.00,A-B,C-D,C-D,C-D,C-D,C-D,6.0
9166,Scheme9167,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,D-A,C-D,D-A,D-A,D-A,6.0


In [27]:
from schemes import *

schemes: set[type[Scheme]] = {
    value for value in globals().values()
    if isinstance(value, type) and issubclass(value, Scheme) and value is not Scheme
    and not value.__name__.startswith("Scheme_")
}

schemes

{schemes.AlphabeticalPriority,
 schemes.AnglophoneTraditional,
 schemes.ChineseTraditional,
 schemes.DoubleSurname,
 schemes.DualLineage,
 schemes.JointFamily,
 schemes.Matrilineal,
 schemes.MatrilinealMarriage,
 schemes.SpanishTraditional,
 schemes.StarWars,
 schemes.Symmetric}

In [28]:
COLUMNS: list[str] = list(Family._fields)
VALUES: list[list[str]] = [list(export_scheme_to_family(scheme)) for scheme in schemes]

manual_schemes: pd.DataFrame = df[df.set_index(COLUMNS).index.isin(VALUES)]
manual_schemes

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
0,Scheme1,0.0,1.00,1.00,1.00,1.00,1.00,0.00,0.00,A,A,B,B,A,A,5.00
1,Scheme2,0.0,0.00,1.00,1.00,1.00,0.00,0.00,1.00,A,A,B,B,A,B,4.00
3,Scheme4,0.0,1.00,1.00,1.00,0.00,0.00,1.00,1.00,A,A,B,B,B,B,5.00
4,Scheme5,1.0,0.50,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.00
5,Scheme6,1.0,0.50,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.00
10,Scheme11,0.5,0.75,1.00,0.75,1.00,0.75,0.50,0.50,A,A,B,B-A,A,A,5.75
3671,Scheme3672,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-C,A-C,5.00
3684,Scheme3685,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-D,A-D,5.00
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25


In [29]:
manual_schemes["manual"] = [[] for i in range(len(manual_schemes))]

for i in manual_schemes.index:
    row_scheme: type[Scheme] = import_scheme_from_dataframe_row_index(manual_schemes, i)
    for scheme in schemes:
        if export_scheme_to_family(scheme) == export_scheme_to_family(row_scheme):
            manual_schemes.loc[i, "manual"].append(scheme.__name__)

manual_schemes

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score,manual
0,Scheme1,0.0,1.00,1.00,1.00,1.00,1.00,0.00,0.00,A,A,B,B,A,A,5.00,[ChineseTraditional]
1,Scheme2,0.0,0.00,1.00,1.00,1.00,0.00,0.00,1.00,A,A,B,B,A,B,4.00,[Symmetric]
3,Scheme4,0.0,1.00,1.00,1.00,0.00,0.00,1.00,1.00,A,A,B,B,B,B,5.00,[Matrilineal]
4,Scheme5,1.0,0.50,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.00,[AnglophoneTraditional]
5,Scheme6,1.0,0.50,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.00,[MatrilinealMarriage]
10,Scheme11,0.5,0.75,1.00,0.75,1.00,0.75,0.50,0.50,A,A,B,B-A,A,A,5.75,[StarWars]
3671,Scheme3672,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-C,A-C,5.00,[SpanishTraditional]
3684,Scheme3685,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-D,A-D,5.00,[DualLineage]
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25,"[AlphabeticalPriority, JointFamily]"


In [32]:
num_schemes_at_least_score: dict[float, int] = {}
for i in range(round(df.score.max() * 4), round(df.score.min() * 4) - 1, -1):
    threshold: float = i / 4 - 0.01
    num_schemes_at_least_score[round(i / 4, 2)] = len(df[df.score >= threshold])

num_schemes_at_least_score

{6.25: 4,
 6.0: 12,
 5.75: 36,
 5.5: 110,
 5.25: 318,
 5.0: 624,
 4.75: 1344,
 4.5: 2158,
 4.25: 3130,
 4.0: 4114,
 3.75: 4990,
 3.5: 6384,
 3.25: 7944,
 3.0: 8942,
 2.75: 9374,
 2.5: 9728,
 2.25: 10272,
 2.0: 10614,
 1.75: 10658,
 1.5: 10666}